# 02 — Preprocessing

### In this notebook, we load, clean, and prepare the raw application and credit datasets.  
### The goal is to obtain a clean and usable dataset for the modeling step.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
path_to_repo = Path('..').resolve()
path_to_data = path_to_repo / 'data'

In [3]:
application = pd.read_csv(path_to_data / 'application_record.csv')
credit = pd.read_csv(path_to_data / 'credit_record.csv')

In [4]:
application = application.drop_duplicates("ID")

In [5]:
application["CODE_GENDER"] = application["CODE_GENDER"].map({"F":0, "M":1})
application["FLAG_OWN_CAR"] = application["FLAG_OWN_CAR"].map({"N":0, "Y":1})
application["FLAG_OWN_REALTY"] = application["FLAG_OWN_REALTY"].map({"N":0, "Y":1})

#### Cleaning key fields
We remove duplicated IDs and convert key binary variables into numerical format.


In [6]:
if "OCCUPATION_TYPE" in application.columns:
    application = application.drop(columns=["OCCUPATION_TYPE"])

In [7]:
application["AGE"] = (-application["DAYS_BIRTH"] / 365).astype(int)

application["EXPERIENCE"] = application["DAYS_EMPLOYED"].apply(
    lambda x: 0 if x > 0 else int(-x/365)
)

In [8]:
application = application.drop(columns=["DAYS_BIRTH","DAYS_EMPLOYED"])

In [9]:
application["CNT_FAM_MEMBERS"] = application["CNT_FAM_MEMBERS"].round().astype(int)

In [10]:
cat_cols = [
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE"
]

for col in cat_cols:
    application[col] = application[col].astype("category").cat.codes

### Core feature transformations
We compute essential features such as age and employment duration from the raw day-based variables.


In [11]:
credit["bad"] = credit["STATUS"].apply(lambda x: 1 if x in ["2","3","4","5"] else 0)

client_status = credit.groupby("ID")["bad"].max().reset_index()

In [12]:
application["LOG_INCOME"] = np.log1p(application["AMT_INCOME_TOTAL"])
application = application.drop(columns=["AMT_INCOME_TOTAL"])

### Target construction
We derive the binary outcome (`bad`) from the client's credit history.


In [13]:
data = application.merge(client_status, on="ID", how="inner")

In [14]:
data.to_csv(path_to_data / "preprocessed_data.csv", index=False)

# Conclusion

Through this preprocessing stage, we cleaned the data, standardized key variables, engineered the main features, and created the target.  
We now have a consistent dataset that we will use for feature engineering and modeling.
